# 03 — Statistical Volatility Models

Naive persistence baselines, OLS on all features, a HAR-style regression, and GARCH(1,1). All evaluation is chronological — random cross-validation would train on the future of its own test points, which badly inflates scores for persistent series.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.figsize": (11, 5), "figure.dpi": 100,
                     "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.float_format", lambda v: f"{v:.4f}")


In [2]:
from src.data import build_market_dataset
from src.features import build_features, feature_columns, log_returns
from src.evaluation import (chronological_split, walk_forward_folds,
                            walk_forward_evaluate, forecast_metrics)
from src.models import NaiveForecaster, LinearModel, HARModel

market = build_market_dataset()
df = build_features(market)
features = feature_columns(df)
X, y = df[features], df["rv_5d_forward"]

split = chronological_split(len(df))
train, val, test = split.slices(df)
folds = walk_forward_folds(len(df), test_start=split.val_end, fold_size=63)
print(f"train {train.index[0].date()}–{train.index[-1].date()}  |  "
      f"val {val.index[0].date()}–{val.index[-1].date()}  |  "
      f"test {test.index[0].date()}–{test.index[-1].date()}  ({len(folds)} folds)")

train 2010-10-18–2021-10-15  |  val 2021-10-18–2024-02-28  |  test 2024-02-29–2026-07-14  (10 folds)


## Naive persistence baselines

"The next 5 days will look like the recent past." Any model worth its complexity must beat this free benchmark.

In [3]:
y_test = y.loc[test.index]
results = {}
for name, col in [("Naive (last 5d RV)", "rv_5d"), ("Naive (last 21d RV)", "rv_21d")]:
    pred = walk_forward_evaluate(lambda c=col: NaiveForecaster(c), X, y, folds)
    results[name] = forecast_metrics(y_test, pred.loc[y_test.index].to_numpy())
pd.DataFrame(results).T

,MAE,RMSE,R2,QLIKE
Naive (last 5d RV),0.0558,0.0940,-0.0435,0.9711
Naive (last 21d RV),0.0573,0.0970,-0.1109,0.5991


## Linear regression on standardized features

In [4]:
pred_lin = walk_forward_evaluate(LinearModel, X, y, folds)
results["Linear Regression"] = forecast_metrics(y_test, pred_lin.loc[y_test.index].to_numpy())

lin = LinearModel().fit(train[features], train["rv_5d_forward"])
lin.coefficients(train[features]).head(10).to_frame("standardized coefficient")

,standardized coefficient
ret_std_21d,-0.2065
vix,0.1731
rv_21d,0.1582
return_21d,-0.1369
ret_mean_21d,0.1277
vix_mean_21d,-0.0667
vix_rv_spread,-0.0501
downside_vol_21d,-0.0277
max_drawdown_63d,0.0206
dist_ma_63d,0.0206


## HAR-style model

Corsi's Heterogeneous Autoregressive model regresses future RV on trailing RV at short (5d), medium (21d), and long (63d) horizons — capturing traders operating at heterogeneous frequencies. Despite being OLS on three regressors, it is a famously strong benchmark.

In [5]:
pred_har = walk_forward_evaluate(HARModel, X, y, folds)
results["HAR"] = forecast_metrics(y_test, pred_har.loc[y_test.index].to_numpy())

har = HARModel().fit(train[features], train["rv_5d_forward"])
print(har.coefficients(train[features]).to_string())
pd.DataFrame(results).T.sort_values("RMSE")

rv_5d    0.0619
rv_21d   0.0116
rv_63d   0.0049


,MAE,RMSE,R2,QLIKE
Linear Regression,0.0478,0.0759,0.3190,0.3976
HAR,0.0495,0.0824,0.1978,0.5150
Naive (last 5d RV),0.0558,0.0940,-0.0435,0.9711
Naive (last 21d RV),0.0573,0.0970,-0.1109,0.5991


## GARCH(1,1)

The classical econometric benchmark: conditional variance follows
$\sigma^2_t = \omega + \alpha r^2_{t-1} + \beta \sigma^2_{t-1}$.
Below we fit once on the training sample to inspect the estimated persistence; the full rolling-refit test-set evaluation is done in `run_pipeline.py` (results in notebook 05).

In [6]:
from arch import arch_model

r_train = log_returns(market["adj_close"]).dropna().loc[:train.index[-1]] * 100
garch = arch_model(r_train, vol="GARCH", p=1, q=1, mean="Constant").fit(disp="off")
print(garch.summary())
alpha, beta = garch.params["alpha[1]"], garch.params["beta[1]"]
print(f"\npersistence alpha + beta = {alpha + beta:.4f}  (close to 1 = shocks decay slowly)")

                     Constant Mean - GARCH Model Results                      
Dep. Variable:              adj_close   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -3676.10
Distribution:                  Normal   AIC:                           7360.19
Method:            Maximum Likelihood   BIC:                           7384.17
                                        No. Observations:                 2967
Date:                Thu, Aug 13 2026   Df Residuals:                     2966
Time:                        05:31:01   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0888  1.312e-02      6.767  1.319e-11 [6.308e-0